# Extended Practice Lab: Multi-Variable Linear Regression — Feature Scaling, Learning Rate & NumPy Vectorization

**Combined from** Coursera Machine Learning (Andrew Ng) C1 W2 labs:
- Python / NumPy / Vectorization
- Multiple Variable Linear Regression
- Feature Scaling and Learning Rate

Use this **skeleton** to practice.  A complete **Solution** notebook is provided separately.


## Purpose of this Lab — What it Helps You Understand and Resolve

**Core purpose**  
This combined lab turns three related ideas into a single, practical workflow:

1. **NumPy vectorization** – replace slow Python loops with fast, readable `np.dot` / broadcasting operations that scale to real data sizes.
2. **Multi-variable linear regression** – extend the single-feature model to several house attributes (size, bedrooms, floors, age) so the model can make realistic price predictions.
3. **Feature scaling + learning-rate choice** – diagnose why gradient descent fails or crawls when feature magnitudes differ by orders of magnitude, and fix it with z-score normalization so a single, larger learning rate works for every parameter.

**Problems it resolves**
- “My gradient descent cost is *increasing*” → almost always an oversized learning rate relative to the raw feature scales.
- “Training takes forever / parameters move at very different speeds” → features with very different ranges (sqft vs number of bedrooms) produce gradients that differ by factors of 100–1000; scaling equalizes them.
- “I can predict only after I retrain everything” → you learn to *store* the training mean and standard deviation and apply the identical transform to any new house.
- “I don’t know which α to pick” → systematic trial of three regimes (too big, borderline, safe) plus the rule-of-thumb that after z-score scaling α ≈ 0.1 is a solid starting point.

**Audience value**
- **Practitioner / data scientist**: concrete, copy-pasteable routines for cost, gradient, and z-score that work with any tabular regression problem.
- **Engineer / technician**: clear diagnostics (cost curves, parameter oscillation plots) that tell you *why* the optimizer is misbehaving.
- **Decision maker**: a quantified demonstration that proper scaling turns a sluggish or divergent algorithm into a reliable pricing tool in a few hundred iterations.

By the end you will be able to take any multi-feature regression data set, scale it, run stable gradient descent, and produce a usable prediction pipeline.


## Analysis Flowchart

```mermaid
flowchart TD
    A[Load / generate housing data<br/>size, bedrooms, floors, age → price] --> B[Explore: scatter each feature vs price]
    B --> C[Implement vectorized predict / cost / gradient]
    C --> D{Try Gradient Descent<br/>without scaling}
    D -->|α too large| E[Cost diverges / oscillates]
    D -->|α too small| F[Cost decreases slowly]
    E --> G[Feature scaling: z-score normalize]
    F --> G
    G --> H[Re-run GD with larger α e.g. 0.1]
    H --> I[Fast convergence, accurate predictions]
    I --> J[Predict new house after normalizing with train μ,σ]
    J --> K[Simulation: vary α, noise, n_features]
```


## Detailed Cheat Sheet (keep open while working the skeleton)

### NumPy vectorization essentials
| Task | Code |
|------|------|
| Create vector | `a = np.array([1,2,3])` or `np.zeros(n)`, `np.arange(n)` |
| Dot product | `np.dot(a, b)`  (preferred over loop) |
| Column mean / std | `mu = X.mean(axis=0)`, `sigma = X.std(axis=0)` |
| Broadcast | `X_norm = (X - mu) / sigma` |
| Peak-to-peak | `np.ptp(X, axis=0)` |

### Multi-variable linear regression
$$
f_{\mathbf{w},b}(\mathbf{x}) = \mathbf{w}\cdot\mathbf{x} + b
$$
$$
J(\mathbf{w},b)=\frac{1}{2m}\sum_{i=0}^{m-1}(f_{\mathbf{w},b}(\mathbf{x}^{(i)})-y^{(i)})^2
$$
$$
w_j := w_j - \alpha\frac{\partial J}{\partial w_j},\quad
b := b - \alpha\frac{\partial J}{\partial b}
$$
$$
\frac{\partial J}{\partial w_j}=\frac{1}{m}\sum_i(f-y)x_j^{(i)},\quad
\frac{\partial J}{\partial b}=\frac{1}{m}\sum_i(f-y)
$$

### Feature scaling (z-score)
$$
x_j^{(i)}\leftarrow\frac{x_j^{(i)}-\mu_j}{\sigma_j}
$$
- Store `mu`, `sigma` from **training** set; reuse for any new example.
- After scaling, learning rate can be much larger (e.g. `0.1`).

### Learning-rate diagnostics
| Symptom | Likely cause | Action |
|---------|--------------|--------|
| Cost **increases** | \(\alpha\) too large | Reduce \(\alpha\) by 3–10× |
| Cost decreases very slowly | \(\alpha\) too small or features unscaled | Scale features and/or increase \(\alpha\) |
| Oscillating parameters | \(\alpha\) near the stability limit | Slightly reduce \(\alpha\) |

### Audience adaptation (from supplied PDFs)
- **Data-literate / technician**: show equations, contour plots, gradient magnitudes.
- **Executive / decision maker**: lead with “scaled features cut training time dramatically and make the model usable for pricing new houses”.
- **Nonspecialist**: use concrete house example (1200 sqft, 3 bed…) and dollar prediction; avoid jargon.


## Goals
- Review NumPy vectorization (dot product, broadcasting) needed for multi-feature models.
- Implement prediction, cost and gradient for multiple features.
- Observe how an inappropriate learning rate \(\alpha\) makes gradient descent diverge or crawl.
- Apply **z-score feature scaling** and re-run gradient descent with a much larger, stable \(\alpha\).
- Make a price prediction for a new house (must normalize with training \(\mu,\sigma\)).
- Explore alternate scalers and a Monte-Carlo simulation of \(\alpha\) / noise effects.


## Step 1 — Imports & reproducibility


In [ ]:
import copy
import math
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

np.random.seed(42)
np.set_printoptions(precision=2, suppress=True)
plt.style.use("seaborn-v0_8-whitegrid")
%matplotlib inline


## Step 2 — Load (or generate) the multi-feature housing data

The data set contains four features: `size_sqft`, `bedrooms`, `floors`, `age` and the target `price_1000s`.


In [ ]:
# TODO: load the CSV that was prepared for this project
df = pd.read_csv("data/housing_multi.csv")
print(df.shape)
print(df.head())
X_train = df[["size_sqft", "bedrooms", "floors", "age"]].values.astype(float)
y_train = df["price_1000s"].values.astype(float)
X_features = ["size_sqft", "bedrooms", "floors", "age"]
print("X shape:", X_train.shape, "y shape:", y_train.shape)


## Step 3 — Explore: each feature vs price
Plot the four scatter plots (share y-axis).  Which feature appears most strongly related to price?


In [ ]:
# TODO: create 1x4 subplot of feature vs price
fig, ax = plt.subplots(1, 4, figsize=(14, 3), sharey=True)
for i in range(4):
    ax[i].scatter(X_train[:, i], y_train, alpha=0.7)
    ax[i].set_xlabel(X_features[i])
ax[0].set_ylabel("Price (1000s)")
fig.suptitle("Feature vs Price (raw data)")
plt.tight_layout()
plt.show()


## Step 4 — Vectorized prediction (NumPy review)

Implement both a pure-Python loop version and a vectorized `np.dot` version.


In [ ]:
def predict_loop(x, w, b):
    """Element-wise prediction (for understanding)."""
    # TODO: loop over features, accumulate w[j]*x[j], add b
    p = 0.0
    for j in range(len(x)):
        p += w[j] * x[j]
    return p + b

def predict(x, w, b):
    """Vectorized prediction."""
    # TODO: return np.dot(x, w) + b
    return np.dot(x, w) + b

# quick sanity check with arbitrary parameters
w_demo = np.array([0.1, 10.0, -5.0, -1.0])
b_demo = 50.0
x0 = X_train[0]
print("loop :", predict_loop(x0, w_demo, b_demo))
print("vector:", predict(x0, w_demo, b_demo))


## Step 5 — Cost function for multiple variables
$$
J(\mathbf{w},b)=\frac{1}{2m}\sum_{i=0}^{m-1}\bigl(f_{\mathbf{w},b}(\mathbf{x}^{(i)})-y^{(i)}\bigr)^2
$$


In [ ]:
def compute_cost(X, y, w, b):
    """
    X: (m,n), y: (m,), w: (n,), b: scalar
    """
    m = X.shape[0]
    # TODO: accumulate squared errors, return mean / 2
    cost = 0.0
    for i in range(m):
        f = np.dot(X[i], w) + b
        cost += (f - y[i]) ** 2
    return cost / (2 * m)

print("Cost at zero weights:", compute_cost(X_train, y_train, np.zeros(4), 0.0))


## Step 6 — Gradient for multiple variables


In [ ]:
def compute_gradient(X, y, w, b):
    """
    Returns dj_db (scalar), dj_dw (n,)
    """
    m, n = X.shape
    dj_dw = np.zeros(n)
    dj_db = 0.0
    # TODO: for each example compute error, accumulate
    for i in range(m):
        err = (np.dot(X[i], w) + b) - y[i]
        for j in range(n):
            dj_dw[j] += err * X[i, j]
        dj_db += err
    dj_dw /= m
    dj_db /= m
    return dj_db, dj_dw

dj_db, dj_dw = compute_gradient(X_train, y_train, np.zeros(4), 0.0)
print("dj_db:", dj_db)
print("dj_dw:", dj_dw)


## Step 7 — Batch gradient descent


In [ ]:
def gradient_descent(X, y, w_in, b_in, alpha, num_iters, cost_fn=compute_cost, grad_fn=compute_gradient):
    w = copy.deepcopy(w_in)
    b = b_in
    J_hist = []
    for i in range(num_iters):
        dj_db, dj_dw = grad_fn(X, y, w, b)
        w = w - alpha * dj_dw
        b = b - alpha * dj_db
        if i < 100000:
            J_hist.append(cost_fn(X, y, w, b))
        if i % max(1, num_iters // 10) == 0:
            print(f"Iter {i:4d}: cost {J_hist[-1]:.4f}")
    return w, b, J_hist


## Step 8 — Learning-rate experiments on *raw* (unscaled) data

Try three values of \(\alpha\).  Record what happens to the cost.


In [ ]:
# TODO: run short trajectories with three alphas
alphas = [9.9e-7, 9e-7, 1e-7]
histories = {}
for a in alphas:
    print(f"\n=== alpha = {a} ===")
    w0 = np.zeros(X_train.shape[1])
    _, _, hist = gradient_descent(X_train, y_train, w0, 0.0, a, 20)
    histories[a] = hist

# plot cost curves
fig, ax = plt.subplots(1, 3, figsize=(14, 3))
for i, a in enumerate(alphas):
    ax[i].plot(histories[a])
    ax[i].set_title(f"α = {a}")
    ax[i].set_xlabel("iteration")
    ax[i].set_ylabel("cost")
plt.tight_layout()
plt.show()


### Observation
- Too-large \(\alpha\): cost increases (divergence / overshoot).
- Borderline \(\alpha\): cost decreases but parameters may oscillate.
- Safe small \(\alpha\): cost decreases slowly; many iterations needed.

The root cause is the huge difference in feature scales (sqft ~ 2000 vs bedrooms ~ 3).


## Step 9 — Z-score feature scaling

Implement
$$
x_j \leftarrow \frac{x_j - \mu_j}{\sigma_j}
$$
and store \(\mu\) and \(\sigma\) for later predictions.


In [ ]:
def zscore_normalize(X):
    """Return X_norm, mu, sigma (all column-wise)."""
    # TODO
    mu = np.mean(X, axis=0)
    sigma = np.std(X, axis=0)
    # avoid division by zero
    sigma = np.where(sigma == 0, 1.0, sigma)
    X_norm = (X - mu) / sigma
    return X_norm, mu, sigma

X_norm, mu, sigma = zscore_normalize(X_train)
print("mu   :", mu)
print("sigma:", sigma)
print("ptp raw :", np.ptp(X_train, axis=0))
print("ptp norm:", np.ptp(X_norm, axis=0))


Visualize the effect of normalization on two features.


In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10, 4))
ax[0].scatter(X_train[:, 0], X_train[:, 3], alpha=0.7)
ax[0].set_xlabel("size_sqft"); ax[0].set_ylabel("age")
ax[0].set_title("Raw (unequal scales)")
ax[0].axis("equal")
ax[1].scatter(X_norm[:, 0], X_norm[:, 3], alpha=0.7)
ax[1].set_xlabel("size (z)"); ax[1].set_ylabel("age (z)")
ax[1].set_title("Z-score normalized")
ax[1].axis("equal")
plt.tight_layout()
plt.show()


## Step 10 — Gradient descent on *scaled* features (large α)


In [ ]:
# TODO: run with alpha = 0.1 for ~1000 iterations
w_norm, b_norm, hist_norm = gradient_descent(
    X_norm, y_train, np.zeros(4), 0.0, alpha=0.1, num_iters=1000
)
print("Final w:", w_norm)
print("Final b:", b_norm)

plt.figure(figsize=(6, 3))
plt.plot(hist_norm)
plt.xlabel("iteration"); plt.ylabel("cost")
plt.title("Cost vs iteration (scaled features, α=0.1)")
plt.show()


## Step 11 — Predictions vs targets (scaled model)


In [ ]:
yp = X_norm @ w_norm + b_norm   # vectorized predictions

fig, ax = plt.subplots(1, 4, figsize=(14, 3), sharey=True)
for i in range(4):
    ax[i].scatter(X_train[:, i], y_train, label="target", alpha=0.6)
    ax[i].scatter(X_train[:, i], yp, label="predict", alpha=0.6)
    ax[i].set_xlabel(X_features[i])
ax[0].set_ylabel("Price (1000s)")
ax[0].legend()
fig.suptitle("Target vs prediction (z-score model)")
plt.tight_layout()
plt.show()


## Step 12 — Predict a new house
A house with 1200 sqft, 3 bedrooms, 1 floor, 40 years old.
**Must** normalize with the training \(\mu,\sigma\).


In [ ]:
x_new = np.array([1200., 3., 1., 40.])
# TODO: normalize then predict
x_new_norm = (x_new - mu) / sigma
price_pred = np.dot(x_new_norm, w_norm) + b_norm
print(f"Predicted price: ${price_pred * 1000:,.0f}")


## Alternate implementations

### A. Min-max scaling instead of z-score
$$
x \leftarrow \frac{x - x_{\min}}{x_{\max}-x_{\min}}
$$


In [ ]:
def minmax_normalize(X):
    X_min = X.min(axis=0)
    X_max = X.max(axis=0)
    rng = np.where(X_max == X_min, 1.0, X_max - X_min)
    return (X - X_min) / rng, X_min, X_max

X_mm, xmin, xmax = minmax_normalize(X_train)
w_mm, b_mm, _ = gradient_descent(X_mm, y_train, np.zeros(4), 0.0, 0.1, 800)
print("minmax final cost:", compute_cost(X_mm, y_train, w_mm, b_mm))


### B. Vectorized gradient (no Python loops over examples)


In [ ]:
def compute_gradient_vectorized(X, y, w, b):
    m = X.shape[0]
    f = X @ w + b
    err = f - y
    dj_dw = (X.T @ err) / m
    dj_db = np.sum(err) / m
    return dj_db, dj_dw

# sanity check
d1, dw1 = compute_gradient(X_norm, y_train, w_norm, b_norm)
d2, dw2 = compute_gradient_vectorized(X_norm, y_train, w_norm, b_norm)
print("db match:", np.isclose(d1, d2))
print("dw match:", np.allclose(dw1, dw2))


### C. Optional scikit-learn StandardScaler (same math)


In [ ]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_sk = scaler.fit_transform(X_train)
print("sklearn mu ~", scaler.mean_)
print("manual  mu =", mu)


## More Practice

1. Re-run the unscaled gradient descent for 5 000 iterations with a very small \(\alpha\) (e.g. 1e-8).  Does cost eventually reach the scaled-model level?
2. Add a fifth synthetic feature (e.g. “distance to city centre”) with a very different scale and repeat the divergence experiment.
3. Implement mean-normalization \( (x-\mu)/(\max-\min) \) and compare final cost and iteration count with z-score.
4. Plot the trajectory of a single weight \(w_0\) for the three raw-\(\alpha\) runs (you will need to record `w` history).


## Simulation Section — explore α, noise and sample size

Change the parameters in the cell below and re-run to see how convergence speed and final cost behave.


In [ ]:
# === SIMULATION CONTROLS (edit these) ===
SIM_ALPHA      = 0.1      # learning rate after scaling
SIM_ITERS      = 500
SIM_NOISE_STD  = 40.0     # noise added to synthetic price
SIM_N_SAMPLES  = 100
SIM_SEED       = 123
# =======================================

rng = np.random.default_rng(SIM_SEED)
size = rng.integers(800, 3500, SIM_N_SAMPLES)
beds = rng.integers(1, 6, SIM_N_SAMPLES)
floors = rng.integers(1, 3, SIM_N_SAMPLES)
age = rng.integers(5, 80, SIM_N_SAMPLES)
price = (50 + 0.12*size + 15*beds - 20*floors - 1.2*age
         + rng.normal(0, SIM_NOISE_STD, SIM_N_SAMPLES))
X_sim = np.column_stack([size, beds, floors, age]).astype(float)
y_sim = np.clip(price, 80, 900)

X_sim_n, mu_s, sig_s = zscore_normalize(X_sim)
w_s, b_s, hist_s = gradient_descent(
    X_sim_n, y_sim, np.zeros(4), 0.0, SIM_ALPHA, SIM_ITERS
)

fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
ax[0].plot(hist_s)
ax[0].set_title(f"Cost (α={SIM_ALPHA}, noise={SIM_NOISE_STD})")
ax[0].set_xlabel("iteration")
ax[1].scatter(y_sim, X_sim_n @ w_s + b_s, alpha=0.7)
ax[1].plot([y_sim.min(), y_sim.max()], [y_sim.min(), y_sim.max()], "r--")
ax[1].set_xlabel("true price"); ax[1].set_ylabel("predicted")
ax[1].set_title("Prediction quality")
plt.tight_layout()
plt.show()
print("Final cost:", hist_s[-1])
print("w:", w_s, "b:", b_s)


## Key Takeaways
- Feature scales that differ by orders of magnitude make a single learning rate unstable; **z-score (or min-max) scaling** equalizes them.
- After scaling, \(\alpha\approx 0.1\) is usually a good starting point for batch gradient descent.
- Always store the training set \(\mu\) and \(\sigma\) (or min/max) and apply the *same* transform to every future example.
- Vectorized NumPy (`X @ w`, `X.T @ err`) is both faster and clearer than explicit Python loops once you understand the shapes.
- Cost curves are the primary diagnostic: rising cost → reduce \(\alpha\); flat cost → increase \(\alpha\) or scale features.
